# Model visualization

This notebook contains some useful functions for visualization purposes.

In [1]:
# pip install shap

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.inspection import permutation_importance
import shap

sns.set(style="whitegrid", font_scale=1.2)

## For all kinds of models:

In [ ]:
def plot_pred_vs_true(model, model_name):
    """
    Plots predicted vs true price values for a given model.
    """

    y_pred = model.predict(X_full_final)

    plt.figure(figsize=(6,6))
    sns.scatterplot(x=y_full, y=y_pred, alpha=0.3, s=15)
    plt.plot([y_full.min(), y_full.max()],
             [y_full.min(), y_full.max()],
             color='red', linewidth=2)

    plt.xlabel("True Price")
    plt.ylabel("Predicted Price")
    plt.title(f"{model_name}: Predicted vs True")
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_residuals(model, model_name):
    """
    Plots residuals vs predicted price values for a given model.
    """

    y_pred = model.predict(X_full_final)
    residuals = y_full - y_pred

    plt.figure(figsize=(7,5))
    sns.scatterplot(x=y_pred, y=residuals, alpha=0.3, s=15)
    plt.axhline(0, color='red', linewidth=2)
    plt.xlabel("Predicted Price")
    plt.ylabel("Residual (True - Pred)")
    plt.title(f"{model_name}: Residuals vs Predicted")
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_residual_distribution(model, model_name):
    """
    Plots the distribution of residuals for a given model.
    """

    y_pred = model.predict(X_full_final)
    residuals = y_full - y_pred

    plt.figure(figsize=(7,5))
    sns.histplot(residuals, kde=True, bins=50, color='steelblue')
    plt.axvline(0, color='red', linewidth=2)
    plt.xlabel("Residual")
    plt.ylabel("Count")
    plt.title(f"{model_name}: Residual Distribution")
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_permutation_importance(model, model_name, n_repeats=5, top=20):
    '''
    Measures how much the model worsens when a feature is randomly shuffled.
    Permutation importance is more trustworthy than feature_importances_.
    '''

    result = permutation_importance(
        model,
        X_full_final,
        y_full,
        n_repeats=n_repeats,
        random_state=42,
        n_jobs=-1
    )

    imp_df = pd.DataFrame({
        "feature": X_full_final.columns,
        "importance": result.importances_mean
    }).sort_values("importance", ascending=False).head(top)

    plt.figure(figsize=(10, top * 0.35 + 1))
    sns.barplot(x="importance", y="feature", data=imp_df, palette="magma")
    plt.title(f"{model_name} - Permutation Importance (Top {top})")
    plt.tight_layout()
    plt.show()

In [7]:
def apply_shap(model, X, model_name="Model", sample_size=2000):
    """
    Applies SHAP to any model by automatically selecting the correct explainer.
    Produces:
        - SHAP summary bar plot (global importance)
        - SHAP beeswarm plot (feature effect distribution)
    """

    # Sample for speed
    X_sample = X.sample(min(sample_size, len(X)), random_state=42)

    # Explainer selection
    if hasattr(model, "tree_") or hasattr(model, "estimators_"):
        # Tree-based models (RF, ExtraTrees, GBM, XGBoost, LGBM, CatBoost)
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)

    elif model.__class__.__name__ in ["LinearRegression", "Ridge", "Lasso", "ElasticNet"]:
        # Linear family
        explainer = shap.LinearExplainer(model, X)
        shap_values = explainer.shap_values(X_sample)

    else:
        # Fallback for SVM, KNN, neural nets, etc.
        background = X.sample(200, random_state=42)
        explainer = shap.KernelExplainer(model.predict, background)
        shap_values = explainer.shap_values(X_sample)

    # PLOTS
    print(f"\nSHAP for {model_name} (sample size = {len(X_sample)})")

    shap.summary_plot(shap_values, X_sample, plot_type="bar", show=True)
    shap.summary_plot(shap_values, X_sample, show=True)

## Linear Models only:

In [ ]:
def plot_coefficients(models, model_names):
    """
    Plots the distribution of coefficients across multiple linear models.
    """

    coef_df = pd.DataFrame({
        name: model.coef_
        for model, name in zip(models, model_names)
    }, index=X_full_final.columns)

    plt.figure(figsize=(12,7))
    coef_df.plot(kind="density", figsize=(12,6))
    plt.title("Distribution of Coefficients Across Linear Models")
    plt.xlabel("Coefficient Value")
    plt.grid(True)
    plt.show()

    # Barplot of absolute coefficient magnitudes 
    top_abs = coef_df.abs().mean(axis=1).sort_values(ascending=False).head(20)
    plt.figure(figsize=(10,6))
    sns.barplot(x=top_abs.values, y=top_abs.index, orient="h")
    plt.title("Most Influential Features (Average Across Models)")
    plt.xlabel("Average |coefficient|")
    plt.tight_layout()
    plt.show()

## Tree-based models only:

In [ ]:
def plot_feature_importance(model, model_name, top=20):
    """
    Plots feature importances.
    """

    importances = model.feature_importances_
    feat_names = X_full_final.columns

    imp_df = pd.DataFrame({
        "feature": feat_names,
        "importance": importances
    }).sort_values("importance", ascending=False).head(top)

    plt.figure(figsize=(10, top * 0.35 + 1))
    sns.barplot(x="importance", y="feature", data=imp_df, palette="viridis")
    plt.title(f"{model_name} - Top {top} Feature Importances")
    plt.tight_layout()
    plt.show()

In [ ]:
def compute_split_frequency(model):
    '''
    Presents how often a feature is used to split. 
    Not as interpretable as features importances, permutable or SHAP, but still good for checking which features drive the tree structure.
    '''

    from collections import Counter
    feature_counts = Counter()

    for tree in model.estimators_:
        tree_ = tree.tree_
        feature = tree_.feature

        for f in feature:
            if f >= 0:  # -2 is "leaf"
                feature_counts[f] += 1

    # Map back to names
    freq_df = pd.DataFrame({
        "feature": [X_full_final.columns[i] for i in feature_counts.keys()],
        "split_count": list(feature_counts.values())
    }).sort_values("split_count", ascending=False)

    return freq_df